In [1]:
import sys
import importlib
sys.path.append("/users/jmduchar/data/jmduchar/Research/mcgill25/rfi_characterization/")
import python.utils as ut

import os
import numpy as np
import arviz as az
from numpy.polynomial.legendre import legvander
import matplotlib.pyplot as plt
from cmdstanpy import CmdStanModel
import json
import glob
from scipy.stats import norm, cauchy, mode, t
from cmdstanpy import from_csv
import seaborn as sns
from tqdm import tqdm
import corner
import json

plt.style.use('seaborn-v0_8')

In [5]:
ABS_DIR = "/users/jmduchar/data/jmduchar/Research/mcgill25/rfi_characterization/"
L_desired = 24
prior_path = ABS_DIR+f"data/json/priors_L{L_desired}.json"
data_path = ABS_DIR+"data/json/legendre_supervised.json"
which_run = "legendre_supervised_p0_L24"

In [6]:
def transition_counts(s, starts):
    C = np.zeros((4,4), dtype=float)
    for t in range(1, len(s)):
        if t in starts: 
            continue
        i, j = s[t-1]-1, s[t]-1
        C[i, j] += 1
    return C

def lognorm_params_from_samples(x, temper=2.0):
    z = np.log(x)
    mu, sd = z.mean(), z.std(ddof=1)
    return mu, sd/temper

def beta_params_from_samples(x, kappa=0.5, eps=0.1):
    m, v = x.mean(), x.var(ddof=1)
    
    # Guard against tiny variance:
    v = max(v, 1e-6 * m*(1-m))
    a = m*(m*(1-m)/v - 1.0)
    b = (1-m)*(m*(1-m)/v - 1.0)
    return a*kappa + eps, b*kappa + eps

In [7]:
outputs = glob.glob(f"../stan/stan_out/{which_run}/*.csv")

fit = from_csv(outputs)

13:55:31 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 1000 iterations at max treedepth (100.0%)
	Chain 2 had 1000 iterations at max treedepth (100.0%)
	Chain 3 had 1000 iterations at max treedepth (100.0%)
	Chain 4 had 1000 iterations at max treedepth (100.0%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


In [13]:
print(fit.diagnose())

Checking sampler transitions treedepth.
4000 of 1000 (400.00%) transitions hit the maximum treedepth limit of 15, or 2^15 leapfrog steps.
Trajectories that are prematurely terminated due to this limit will result in slow exploration.
For optimal performance, increase this limit.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

The following parameters had rank-normalized split R-hat greater than 1.01:
  rate_rising, mu_X[2], mu_X[10], mu_X[13], mu_X[17], alpha_X[6], beta_X, X_sup[5,1], X_sup[14,1], X_sup[26,1], X_sup[4,2], X_sup[5,2], X_sup[14,2], X_sup[20,2], X_sup[26,2], X_sup[4,3], X_sup[5,3], X_sup[13,3], X_sup[27,3], X_sup[34,3], X_sup[1,4], X_sup[3,4], X_sup[5,4], X_sup[40,4], X_sup[4,5], X_sup[13,5], X_sup[26,5], X_sup[4,6], X_sup[5,6], X_sup[14,6], X_sup[26,6], X_sup[4,7], X_sup[5,7], X_sup

In [5]:
# Load JSON data
with open(
    data_path,
    "r"
) as f:
    data_dict = json.load(f)
    
post = fit.stan_variables()
start_idx = data_dict['start_idx_sup']
s = data_dict['s_sup']

In [6]:
C = transition_counts(s, set(start_idx))

# tampering
lam = 0.5

# weak prior
eta_clean  = np.array([9.0, 1.0, 0.2])
eta_rising = np.array([8.0, 2.0, 0.2])
eta_decay  = np.array([5.0, 0.5, 4.5, 0.2])
eta_blip   = np.array([8.0, 0.5, 0.5, 0.1])

alpha_clean  = eta_clean  + lam * C[0, [0,1,3]]
alpha_rising = eta_rising + lam * C[1, [1,2,3]]
alpha_decay  = eta_decay  + lam * C[2, [0,1,2,3]]
alpha_blip   = eta_blip   + lam * C[3, [0,1,2,3]]

rr_log_mu, rr_log_sigma = lognorm_params_from_samples(post["rate_rising"], temper=2.0)
sig_log_mu, sig_log_sigma = lognorm_params_from_samples(post["sigma"], temper=2.0)
k_log_mu,  k_log_sigma  = lognorm_params_from_samples(post["k_blip"], temper=2.0)

rd_alpha, rd_beta = beta_params_from_samples(post["rate_decay"], kappa=0.5, eps=0.1)

mu_blip_mean = float(np.mean(post["mu_blip"]))
mu_blip_sd   = float(np.std(post["mu_blip"], ddof=1) * 2.0)

In [7]:
L_previous = len(post["mu_X"].mean(axis=0))

if L_desired == L_previous:
    print("Initializing Legendre coefficient priors based on previous run.")
    mu_X_mean = post["mu_X"].mean(axis=0)
    mu_X_sd   = post["mu_X"].std(axis=0, ddof=1) * 2.0
    alpha_X_log_mu, alpha_X_log_sigma = np.log(post["alpha_X"]).mean(axis=0), np.log(post["alpha_X"]).std(axis=0, ddof=1)/2
    beta_X_log_mu, beta_X_log_sigma   = np.log(post["beta_X"]).mean(), max(np.log(post["beta_X"]).std(ddof=1)/2, 0.1)
    
else:
    print("Initializing Legendre coefficient priors from scratch.")
    
    mu_X_mean = np.full(L_desired, -1.0, dtype=float)
    mu_X_sd   = np.full(L_desired, 3.0 * 2.0, dtype=float)

    alpha_X_log_mu    = np.full(L_desired, np.log(70.0), dtype=float)
    alpha_X_log_sigma = np.full(L_desired, 100.0, dtype=float)

    beta_X_log_mu    = float(np.log(2.0))
    beta_X_log_sigma = float(1)

Initializing Legendre coefficient priors based on previous run.


In [8]:
prior_dict = dict(
    alpha_clean       = alpha_clean.tolist(), 
    alpha_rising      = alpha_rising.tolist(),
    alpha_decay       = alpha_decay.tolist(), 
    alpha_blip        = alpha_blip.tolist(),
    rr_log_mu         = rr_log_mu.tolist(), 
    rr_log_sigma      = rr_log_sigma.tolist(),
    rd_alpha          = rd_alpha.tolist(), 
    rd_beta           = rd_beta.tolist(),
    sig_log_mu        = sig_log_mu, 
    sig_log_sigma     = sig_log_sigma,
    mu_blip_mean      = mu_blip_mean, 
    mu_blip_sd        = mu_blip_sd,
    k_blip_log_mu     = k_log_mu.tolist(), 
    k_blip_log_sigma  = k_log_sigma.tolist(),
    mu_X_mean         = mu_X_mean.tolist(), 
    mu_X_sd           = mu_X_sd.tolist(),
    alpha_X_log_mu    = alpha_X_log_mu.tolist(), 
    alpha_X_log_sigma = alpha_X_log_sigma.tolist(),
    beta_X_log_mu     = beta_X_log_mu, 
    beta_X_log_sigma  = beta_X_log_sigma,
)

In [10]:
with open(
    prior_path,
    "w"
) as f:
    json.dump(prior_dict, f, indent=2)

In [6]:
prior_dict = dict(
    # --- transition priors ---
    alpha_clean       = [10.0, 1.0, 1.0],        # clean -> {clean, rising, blip}
    alpha_rising      = [10.0, 5.0, 1.0],        # rising -> {rising, decay, blip}
    alpha_decay       = [5.0, 1.0, 10.0, 1.0],   # decay  -> {clean, rising, decay, blip}
    alpha_blip        = [10.0, 1.0, 1.0, 1.0],   # blip   -> {clean, rising, decay, blip}

    # --- dynamic parameters ---
    rr_log_mu         = 0.0,        # lognormal mean for rate_rising (exp(0)=1)
    rr_log_sigma      = 1.0,        # wide
    rd_alpha          = 2.0,        # beta(2,2) near-uniform
    rd_beta           = 2.0,

    # --- noise variance ---
    sig_log_mu        = 0.0,        # mean of log(sigma)
    sig_log_sigma     = 2.0,        # covers sigma ~ [0.05, 50]

    # --- blip emission ---
    mu_blip_mean      = 0.0,        # centered
    mu_blip_sd        = 10.0,       # very wide
    k_blip_log_mu     = 0.0,        # lognormal mean for k_blip
    k_blip_log_sigma  = 2.0,        # broad spread

    # --- legendre hyperparameters (lenient, scale-invariant) ---
    mu_X_mean         = [0.0] * L_desired,              # zero-centered
    mu_X_sd           = [5.0] * L_desired,              # wide (allows large coeffs)
    alpha_X_log_mu    = [0.0] * L_desired,              # lognormal mean
    alpha_X_log_sigma = [1.0] * L_desired,              # wide dispersion
    beta_X_log_mu    = float(np.log(2.0)),   # median(beta_X) = 2
    beta_X_log_sigma = 1.0
)